# Linear SVM from Scratch (Portfolio Sample)
This notebook trains a **soft-margin linear SVM** from scratch on synthetic data and compares with scikit-learn.
It also includes an **RBF Gram matrix** demo for intuition.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.svm import LinearSVC

from src.svm_scratch import LinearSVMScratch

# Generate a (mostly) linearly separable dataset
X, y = make_classification(n_samples=600, n_features=2, n_redundant=0, n_informative=2,
                           n_clusters_per_class=1, class_sep=1.5, random_state=42)
y = np.where(y==1, 1, -1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

sc = StandardScaler()
X_train_s = sc.fit_transform(X_train)
X_test_s  = sc.transform(X_test)
X_train_s.shape, X_test_s.shape


In [ ]:

# Train our scratch SVM
svm = LinearSVMScratch(C=1.0, lr=1e-2, max_iter=5000, tol=1e-7, random_state=42).fit(X_train_s, y_train)
preds = svm.predict(X_test_s)

print('Scratch SVM accuracy:', round(accuracy_score(y_test, preds), 3))
print(confusion_matrix(y_test, preds))
print(classification_report(y_test, preds, target_names=['-1','+1']))


In [ ]:

# Compare with scikit-learn LinearSVC
ref = LinearSVC(C=1.0, random_state=42, max_iter=5000)
ref.fit(X_train_s, y_train)
ref_preds = ref.predict(X_test_s)
print('LinearSVC accuracy:', round(accuracy_score(y_test, ref_preds), 3))
print(confusion_matrix(y_test, ref_preds))
print(classification_report(y_test, ref_preds, target_names=['-1','+1']))


In [ ]:

# Plot decision boundary for scratch SVM
def plot_decision_boundary(model, X, y, scaler=None, title='Decision boundary'):
    x_min, x_max = X[:,0].min()-1, X[:,0].max()+1
    y_min, y_max = X[:,1].min()-1, X[:,1].max()+1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
    grid = np.c_[xx.ravel(), yy.ravel()]
    grid_s = scaler.transform(grid) if scaler is not None else grid
    zz = model.decision_function(grid_s).reshape(xx.shape)

    plt.figure(figsize=(6,5))
    plt.contourf(xx, yy, np.sign(zz), alpha=0.2, levels=[-1,0,1])
    plt.contour(xx, yy, zz, levels=[-1,0,1], linestyles=['--','-','--'])
    plt.scatter(X[y==1,0], X[y==1,1], s=12, label='+1')
    plt.scatter(X[y==-1,0], X[y==-1,1], s=12, label='-1')
    plt.legend(); plt.xlabel('x1'); plt.ylabel('x2'); plt.title(title)
    plt.show()

plot_decision_boundary(svm, X_test, y_test, scaler=sc, title='Scratch Linear SVM')


In [ ]:

# Educational: RBF (Gaussian) Gram matrix for a tiny subset
def rbf_gram(X, sigma=1.0):
    # K_ij = exp(-||xi-xj||^2 / (2 sigma^2))
    dists = np.sum((X[:,None,:] - X[None,:,:])**2, axis=2)
    return np.exp(-dists / (2 * sigma**2))

subset = X_train[:50]
K = rbf_gram(subset, sigma=1.5)
print('Gram shape:', K.shape)

plt.figure(figsize=(5,4))
plt.imshow(K, interpolation='nearest')
plt.title('RBF Gram Matrix (sigma=1.5)')
plt.colorbar(); plt.tight_layout(); plt.show()
